In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt

import algorithms.kernelDL_multistep as kdl
from algorithms.dmd import eDMD, rffEDMD
import utils.kernels as kernels
import utils.helpers as helpers
import utils.plotting_params


# koopmanAE
from koopmanAE.model import koopmanAE
from koopmanAE.train import train
from koopmanAE.tools import *
from koopmanAE.read_dataset import pendulum


%load_ext autoreload
%autoreload 2

Connected to a GPU


### general parameters -- change accordingly

In [92]:
theta = 2.4
noise_ratio = 0.1

steps, steps_back, backward, epochs = 8, 8, 1, 600   # koopmanAE
steps_edmdkdl, n_iters = 8, 600                      # EDMD-kDL
embed_dim = 6

### generate train-test data

In [93]:
X_train, X_test, X_train_clean, X_test_clean, noise_scale, a,b = pendulum(noise_ratio=noise_ratio, theta=theta)
dim = X_train.shape[1]

In [94]:
noise_scale.item()

0.28363412802029797

### consistent Koopman AE

In [95]:
seed = 0
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.cuda.manual_seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)
set_seed(seed)
device = get_device()

batchsize=64; init_scale=0.99; alpha_=1
lr=1e-2; wd=0; lamb=1; lr_decay=0.2; lr_update=[30, 200, 400, 500];
nu=1e-1; eta=1e-2; gradclip=0.05; 


#****************************************************************
#     prepare dataset for koopmanAE
#****************************************************************
X_train_kae = add_channels(X_train)

# create dataloader
X_train_kae = torch.from_numpy(X_train_kae).float().contiguous()
trainDat = []
start = 0
for i in np.arange(steps,-1, -1):
    if i == 0:
        trainDat.append(X_train_kae[start:].float())
    else:
        trainDat.append(X_train_kae[start:-i].float())
    start += 1

train_data = torch.utils.data.TensorDataset(*trainDat)
del(trainDat)

train_loader = torch.utils.data.DataLoader(dataset = train_data, batch_size = batchsize, shuffle = True)

#*************************************************************
#             Model
#*************************************************************
model = koopmanAE(dim, 1, embed_dim, steps, steps_back, alpha_, init_scale)
model = model.to(device)

print('**** Setup ****')
print('Total params: %.2fM' % (sum(p.numel() for p in model.parameters())/1000000.0))
print('Total params: %.2fk' % (sum(p.numel() for p in model.parameters())/1000.0))
print('************')
print(model)

#==============================================================================
# Training
#==============================================================================
model, optimizer, epoch_hist = train(model, train_loader, lr=lr, weight_decay=wd, lamb=lamb, num_epochs = epochs,
                    learning_rate_change=lr_decay, epoch_update=lr_update,nu = nu, eta = eta, backward=backward,
                    steps=steps, steps_back=steps_back, gradclip=gradclip)

Connected to a GPU
**** Setup ****
Total params: 0.00M
Total params: 0.91k
************
koopmanAE(
  (encoder): encoderNet(
    (tanh): Tanh()
    (fc1): Linear(in_features=2, out_features=16, bias=True)
    (fc2): Linear(in_features=16, out_features=16, bias=True)
    (fc3): Linear(in_features=16, out_features=6, bias=True)
  )
  (dynamics): dynamics(
    (dynamics): Linear(in_features=6, out_features=6, bias=False)
  )
  (backdynamics): dynamics_back(
    (dynamics): Linear(in_features=6, out_features=6, bias=False)
  )
  (decoder): decoderNet(
    (tanh): Tanh()
    (fc1): Linear(in_features=6, out_features=16, bias=True)
    (fc2): Linear(in_features=16, out_features=16, bias=True)
    (fc3): Linear(in_features=16, out_features=2, bias=True)
  )
)
Connected to a GPU
********** Epoche 1 **********
loss identity:  0.3456119894981384
loss backward:  1.3995094299316406
loss consistent:  6.445886611938477
loss forward:  1.2411494255065918
loss sum:  1.7911713123321533
********** Epoche 

### Standard Koopman AE

In [96]:
seed = 0
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.cuda.manual_seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)
set_seed(seed)

#*************************************************************
#             Model
#*************************************************************
nu=0.0; eta=0.0; backward=0; steps_back=0
model_kae = koopmanAE(dim, 1, embed_dim, steps, steps_back, alpha_, init_scale)
model_kae = model_kae.to(device)

print('**** Setup ****')
print('Total params: %.2fM' % (sum(p.numel() for p in model_kae.parameters())/1000000.0))
print('Total params: %.2fk' % (sum(p.numel() for p in model_kae.parameters())/1000.0))
print('************')
print(model_kae)

#==============================================================================
# Training
#==============================================================================
model_kae, optimizer, epoch_hist = train(model_kae, train_loader, lr=lr, weight_decay=wd, lamb=lamb, num_epochs = epochs,
                    learning_rate_change=lr_decay, epoch_update=lr_update,nu = nu, eta = eta, backward=backward,
                    steps=steps, steps_back=steps_back, gradclip=gradclip)

**** Setup ****
Total params: 0.00M
Total params: 0.91k
************
koopmanAE(
  (encoder): encoderNet(
    (tanh): Tanh()
    (fc1): Linear(in_features=2, out_features=16, bias=True)
    (fc2): Linear(in_features=16, out_features=16, bias=True)
    (fc3): Linear(in_features=16, out_features=6, bias=True)
  )
  (dynamics): dynamics(
    (dynamics): Linear(in_features=6, out_features=6, bias=False)
  )
  (backdynamics): dynamics_back(
    (dynamics): Linear(in_features=6, out_features=6, bias=False)
  )
  (decoder): decoderNet(
    (tanh): Tanh()
    (fc1): Linear(in_features=6, out_features=16, bias=True)
    (fc2): Linear(in_features=16, out_features=16, bias=True)
    (fc3): Linear(in_features=16, out_features=2, bias=True)
  )
)
Connected to a GPU
********** Epoche 1 **********
loss identity:  0.324846088886261
loss forward:  1.2135623693466187
loss sum:  1.5384085178375244
********** Epoche 21 **********
loss identity:  0.009690430015325546
loss forward:  0.04516248777508736
loss 

### EDMD-kDL

In [97]:
X, Y = X_train[:-1,:].T, X_train[1:,:].T
#============== collocation points =================
mc, seed = 8, 42
l_bounds = [X.min() -1.]*dim 
u_bounds = [X.max() + 1.]*dim 
Z = helpers.generate_collocation_points(dim, l_bounds, u_bounds, m=mc, seed=seed).T

#============== model instatiation ==================
sigma, p, gamma = .75, 6, 1e-6
kernel = kernels.gaussianKernel(sigma=sigma) 
edmd_kdl = kdl.EDMDkDL(X=X, Y=Y, Z=Z, kernel=kernel, p=p, gamma=gamma, decoder=True,seed=seed)

#============= training ============================
history_size = 20
edmd_kdl.optimize(n_iters=1000, steps=steps_edmdkdl, beta1=1, beta2=1.5, beta3=1e-6, verbose_after=20,
                  line_search_fn='strong_wolfe', history_size=history_size)

step   0 | loss 5.7e+04
step  20 | loss 4.6e-03
step  40 | loss 7.8e-04
step  60 | loss 2.4e-04
step  80 | loss 1.6e-04
step 100 | loss 1.3e-04
step 120 | loss 1.1e-04
step 140 | loss 8.8e-05
step 160 | loss 7.5e-05
step 180 | loss 6.6e-05
step 200 | loss 6.0e-05
step 220 | loss 5.4e-05
step 240 | loss 5.0e-05
step 260 | loss 4.5e-05
step 280 | loss 4.1e-05
step 300 | loss 3.8e-05
step 320 | loss 3.6e-05
step 340 | loss 3.3e-05
step 360 | loss 3.0e-05
step 380 | loss 2.8e-05
step 400 | loss 2.6e-05
step 420 | loss 2.5e-05
step 440 | loss 2.4e-05
step 460 | loss 2.2e-05
step 480 | loss 2.1e-05
step 500 | loss 2.0e-05
step 520 | loss 1.9e-05
step 540 | loss 1.8e-05
step 560 | loss 1.8e-05
step 580 | loss 1.8e-05
step 600 | loss 1.8e-05
step 620 | loss 1.8e-05
step 640 | loss 1.8e-05
step 660 | loss 1.8e-05
step 680 | loss 1.8e-05
step 700 | loss 1.8e-05
step 720 | loss 1.8e-05
step 740 | loss 1.8e-05
step 760 | loss 1.8e-05
step 780 | loss 1.8e-05
step 800 | loss 1.8e-05
step 820 | loss 

### DMD

In [98]:
dmd = eDMD(X, Y)
dmd.train()

### RFF-EDMD

In [99]:
edmd_rff = rffEDMD(X, Y, sigma=sigma, decoder=True)
edmd_rff.train(n_features=100)

### Testing

In [100]:
M = 1000
x0 = X_test[0]
x0_kae = torch.from_numpy(add_channels(X_test)[0]).contiguous().float()

In [101]:
# koopmanAE
with torch.no_grad():
    u0 = model_kae.encoder(x0_kae.to(device))
    U = [u0]
    for i in range(M-1):
        U.append(model_kae.dynamics(U[i]))
    U = torch.cat(U)
    X_kae= model_kae.decoder(U).squeeze().T.cpu().numpy()

# consistent koopmanAE
with torch.no_grad():
    u0 = model.encoder(x0_kae.to(device))
    U = [u0]
    for i in range(M-1):
        U.append(model.dynamics(U[i]))
    U = torch.cat(U)
    X_ckae= model.decoder(U).squeeze().T.cpu().numpy()
    
# edmd-kDL
with torch.no_grad():
    u0 = edmd_kdl.psi_fn(x0).squeeze()
    U = [u0]
    for i in range(M-1):
        U.append(edmd_kdl.learned_linear_model(U[i]))
    U = np.array(U).T
    X_edmdkdl = edmd_kdl.decoder_fn(U) if edmd_kdl.decoder else U[:X.shape[0],:]
X_edmdkdl = X_edmdkdl.numpy() if isinstance(X_edmdkdl, torch.Tensor) else X_edmdkdl

# dmd
X_dmd = [x0]
for i in range(M-1):
    X_dmd.append(dmd.linear_model(X_dmd[i]))
X_dmd = np.array(X_dmd).T


# edmd-rff
X_edmdrff = [x0]
for i in range(M-1):
    X_edmdrff.append(edmd_rff.predict(X_edmdrff[i]).squeeze())
X_edmdrff = np.array(X_edmdrff).T


**phase portraits plot**

In [102]:
# rescale
X_kae = (X_kae + 1) * (b/2) + a
X_ckae = (X_ckae + 1) * (b/2) + a
X_edmdkdl = (X_edmdkdl + 1) * (b/2) + a
X_edmdrff = (X_edmdrff + 1) * (b/2) + a

In [103]:
fig, ax = plt.subplots(1, 4, figsize=(5.75, 1.5), sharex=True, sharey=True, constrained_layout=True)

for i in range(len(ax)):
    ax[i].plot(X_test_clean[:M, 0], X_test_clean[:M, 1],  lw=0.75, color='black',  label="Ground truth")
    if i == 0:
        ax[i].plot(X_edmdrff[0, :], X_edmdrff[1, :], lw=0.75, color='magenta', label="EDMD-RFF")
    elif i == 1:
        ax[i].plot(X_kae[0, :], X_kae[1, :], lw=0.75, color='orange', label="KAE")
    elif i == 2:
        ax[i].plot(X_ckae[0, :], X_ckae[1, :], lw=0.75, color='darkred', label="cKAE")
    else:
        ax[i].plot(X_edmdkdl[0, :], X_edmdkdl[1, :], lw=0.75, color='darkblue', label="EDMD-kDL")
    ax[i].set_xlabel(r'$x_1$', fontsize=11)
    if i == 0:
        ax[i].set_ylabel(r'$x_2$', fontsize=11)
    ax[i].tick_params(axis='both', labelsize=11)
    ax[i].grid(False)

# collect unique handles/labels from axes
handles_list, labels_list = [], []
for aa in ax:
    h, l = aa.get_legend_handles_labels()
    for hh, ll in zip(h, l):
        if ll not in labels_list:
            handles_list.append(hh); labels_list.append(ll)

# add shared legend on top and save
fig.legend(handles_list, labels_list, loc='upper center', bbox_to_anchor=(0.52, 1.15), ncol=len(labels_list),
            fontsize=11, frameon=False, markerscale=1, columnspacing=.8, handletextpad=0.3, handlelength=1.)

fig.savefig(f"figures/pendulum/undamped_pendulum_theta_{theta}_noise_{noise_ratio}_phase_portraits.pdf", format="pdf", bbox_inches="tight")
plt.close(fig)

**prediction errors**

In [104]:
fig = plt.figure(figsize=(4.5,1.5))
plt.plot(np.linalg.norm(X_test_clean[:M,:].T - X_edmdrff, axis=0) / np.linalg.norm(X_test_clean[:M,:], axis=1), lw=0.75, color='magenta', label='EDMD-RFF')
plt.plot(np.linalg.norm(X_test_clean[:M,:].T - X_kae, axis=0) / np.linalg.norm(X_test_clean[:M,:], axis=1), lw=0.75, color='orange', label='KAE')
plt.plot(np.linalg.norm(X_test_clean[:M,:].T - X_ckae, axis=0) / np.linalg.norm(X_test_clean[:M,:], axis=1), lw=0.75, color='darkred', label='cKAE')
plt.plot(np.linalg.norm(X_test_clean[:M,:].T - X_edmdkdl, axis=0) / np.linalg.norm(X_test_clean[:M,:], axis=1), lw=0.75, color='darkblue', label='EDMD-kDL')
plt.xlabel('Time step', fontsize=11)
plt.ylabel('Prediction error', fontsize=11)
plt.tick_params(axis='both', labelsize=11)
plt.legend(loc = "upper left", ncols=4, bbox_to_anchor=(0., 1.2), fontsize=11, columnspacing=0.8, handlelength=1., handletextpad=0.2, frameon=False)
plt.grid(visible=False)

fig.savefig(f"figures/pendulum/undamped_pendulum_theta_{theta}_noise_{noise_ratio}_prediction_error.pdf", format="pdf", bbox_inches="tight")
plt.close(fig)

In [105]:
fig, axes = plt.subplots(2, 2, figsize=(5.5, 2.), sharex=True, sharey=True, constrained_layout=True)

datasets = [X_edmdrff, X_kae, X_ckae, X_edmdkdl]
labels = ['EDMD-RFF', 'KAE', 'cKAE', 'EDMD-kDL']
colors = ['magenta', 'orange', 'darkred', 'darkblue']

axes = axes.flatten()

for i, ax in enumerate(axes):
    ax.plot(np.linalg.norm(X_test_clean[:M,:].T - datasets[i], axis=0) / np.linalg.norm(X_test_clean[:M,:], axis=1), lw=0.75, color=colors[i], label=labels[i])
    ax.legend(fontsize=16, frameon=False, bbox_to_anchor=(0., 1.2), loc="upper left",  handlelength=1., handletextpad=0.2)        
    ax.tick_params(axis='both', labelsize=16)
    ax.grid(False)
fig.text(-0.05, 0.5, 'Prediction error', va='center', rotation='vertical', fontsize=14)
fig.text(0.5, -0.1, 'Time step', ha='center', fontsize=16)
fig.savefig(f"figures/pendulum/undamped_pendulum_theta_{theta}_noise_{noise_ratio}_prediction_error_multiple.pdf", format="pdf", bbox_inches="tight")
plt.close(fig)


**trajectories**

In [106]:
fig, axes = plt.subplots(2, 1, figsize=(5.5, 2.5), sharex=True, constrained_layout=True)

datasets = [X_test_clean[:M,:].T, X_edmdrff, X_kae, X_ckae, X_edmdkdl]
labels = ['Ground Truth', 'EDMD-RFF', 'KAE', 'cKAE', 'EDMD-kDL']
colors = ['black', 'magenta', 'orange', 'darkred', 'darkblue']

axes = axes.flatten()
# zoom-in the final part
t0 = 750
for i, ax in enumerate(axes):
    for j in range(len(datasets)):
        ax.plot(np.arange(t0, 1000), datasets[j][i,t0:], lw=0.75, color=colors[j], label=labels[j])
        ax.set_xlabel('Time step', fontsize=11)    
        ax.set_ylabel(rf'$x_{{{i+1}}}$', fontsize=11)      
        ax.tick_params(axis='both', labelsize=11)
        ax.grid(False)

# common legend
handles_list, labels_list = [], []
for ax in axes:
    h, l = ax.get_legend_handles_labels()
    for hh, ll in zip(h, l):
        if ll not in labels_list:
            handles_list.append(hh); labels_list.append(ll)
fig.legend(handles_list, labels_list, loc='upper center', bbox_to_anchor=(0.52, 1.15), ncol=len(labels_list),
            fontsize=11, frameon=False, markerscale=1, columnspacing=.8, handletextpad=0.3, handlelength=1.)

fig.savefig(f"figures/pendulum/undamped_pendulum_theta_{theta}_noise_{noise_ratio}_trajectories.pdf", format="pdf", bbox_inches="tight")
plt.close(fig)
